In [4]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape, Polygon
import numpy as np
from pathlib import Path
import os
import sys

In [5]:
CURRENT_DIRECTORY =  os.getcwd()
PARENT_DIRECTORY = os.path.dirname(CURRENT_DIRECTORY)
print(PARENT_DIRECTORY)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src


In [13]:
BASE_DATASET_PATH = Path(PARENT_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


## Obtaining the OSM file of Singapore
Download malaysia map from here: https://download.geofabrik.de/asia/malaysia-singapore-brunei.html# -> raw directory index.

Download the .osm.pbf for 2014 to 2021 and save the files in a folder for each year (eg: 2019_geospatial)

Running in datasets/geospatial_data directory

Run this once, .poly file is used to extract singapore from the OSM file
```
curl -L -o singapore.poly "https://polygons.openstreetmap.fr/get_poly.py?id=536780"
```
Run this for each year (2014-2021)
```
cd <year>_geospatial
osmium extract -p ../singapore.poly -s complete_ways --overwrite -o singapore.osm.pbf malaysia-singapore-brunei-<year>0101.osm.pbf
```
eg: `osmium extract -p singapore.poly -s complete_ways --overwrite -o singapore.osm.pbf malaysia-singapore-brunei-190101.osm.pbf`

Convert `.osm.pbf` file to `.gpkg` file and to EPSG: 3414
```
ogr2ogr -f GPKG singapore.gpkg singapore.osm.pbf
ogr2ogr -f GPKG singapore_3414.gpkg singapore.gpkg -t_srs EPSG:3414
```
Separate lines and multipolygons into 2 different files
```
ogr2ogr -f GPKG singapore_<year>_multipolygons.gpkg singapore_3414.gpkg multipolygons
ogr2ogr -f GPKG singapore_<year>_lines.gpkg singapore_3414.gpkg lines
```
Clean up
```
rm -f singapore.osm.pbf singapore.gpkg singapore_3414.gpkg
```


In [ ]:
QGIS_PATH = '/Applications/QGIS-LTR.app/Contents/Resources/python' 

# Add the QGIS Python path to the system path
sys.path.append(QGIS_PATH)
# Add path to plugins directory (usually needed for 'processing' module)
sys.path.append(os.path.join(QGIS_PATH, 'plugins'))

# QgsApplication.setPrefixPath() tells QGIS where to look for its core C++ libraries (plugins, resources, etc.).
# The first argument is the QGIS installation prefix (usually the folder containing 'bin', 'apps', etc.)
# For your Mac example, we use the path leading up to 'Resources'.
qgis_app_prefix = '/Applications/QGIS-LTR.app/Contents/Resources' # Adjust as needed

print(f"1. Setting QGIS prefix path to: {qgis_app_prefix}")
from qgis.core import QgsApplication
QgsApplication.setPrefixPath(qgis_app_prefix, True)

# Create an instance of QgsApplication. The second argument (False) means no GUI.
# This prepares the environment for non-GUI (standalone) scripting.
qgs = QgsApplication([], False) 

# Load Providers and registries (REQUIRED)
print("2. Initializing QGIS application...")
qgs.initQgis() 
print("QGIS application initialized successfully.")

In [17]:
# # Import QGIS Libraries and Processing
# try:
#     # Now that QGIS is initialized, we can safely import these modules
#     from qgis.analysis import QgsNativeAlgorithms
#     import processing
#     from processing.core.Processing import Processing
#     from qgis.core import QgsProject, QgsProcessingFeedback, QgsVectorLayer, QgsCoordinateReferenceSystem
#     from qgis.PyQt.QtCore import QVariant 
# except ImportError as e:
#     print(f"\n--- FATAL ERROR: MODULE IMPORT FAILED AFTER INITIALIZATION ---")
#     print(f"Please double-check the QGIS_PATH and qgis_app_prefix variables.")
#     print(f"Original error: {e}")
#     sys.exit(1)

In [18]:
### Uncomment to run qgis related commands
# # initialise the Processing Framework
# Processing.initialize()
# # Add the QGIS native algorithms provider
# QgsApplication.processingRegistry().addProvider(QgsNativeAlgorithms())

In [19]:
# reading singapore 2021 multipolygon map data
singapore_2021_filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/singapore_2021_multipolygons.gpkg"
singapore_2021_gpd = gpd.read_file(singapore_2021_filepath)
singapore_2021_gpd.crs

<Projected CRS: EPSG:3414>
Name: SVY21 / Singapore TM
Axis Info [cartesian]:
- N[north]: Northing (metre)
- E[east]: Easting (metre)
Area of Use:
- name: Singapore - onshore and offshore.
- bounds: (103.59, 1.13, 104.07, 1.47)
Coordinate Operation:
- name: Singapore Transverse Mercator
- method: Transverse Mercator
Datum: SVY21
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [ ]:
def read_multipolygon_file(year, BASE_DATASET_PATH):
    """
    Reads geospatial package file

    Parameters
    ------
    year : str
        year of the geospatial data you want to read
    BASE_DATASET_PATH : pathlib.PosixPath
        the base filepath pointing to the "dataset" parent directory
    
    returns
    ------
    singapore_gpd : dataframe
        the geopackage dataframe of singapore's geospatial data
    """
    folder_name = "geospatial_data/" + str(year) + "_geospatial/singapore_" + str(year) + "_multipolygons.gpkg"
    singapore_filepath = BASE_DATASET_PATH / folder_name
    singapore_gpd = gpd.read_file(singapore_filepath)
    print(singapore_gpd.head())
    return singapore_gpd

In [15]:
def create_manual_tags_column(df):
    """
    Create a manual_tags column to the geodataframe as the commercial value in the landuse column is not descriptive enough

    Parameters
    ------
    df : Geodataframe
        The geodataframe of interest

    Returns
    ------
    df : Geodataframe
        A modified version of the input dataframe
    """
    # add a manual_tags column
    df["manual_tags"] = None

    # commercial column is not descriptive enough. 
    # rows where commercial != null AND other_tags contains HDB will be tagged with street_shops
    # rows where commerical != null AND shop == "mall" will be tagged with mall
    singapore_landuse = df[df["landuse"] == "commercial"].copy()

    # conditions on how specific landuse will be classified
    hdb_condition = singapore_landuse["other_tags"].str.contains("HDB", na = False)
    mall_condition = singapore_landuse["shop"] == "mall"

    singapore_landuse[["other_tags", "shop"]] = singapore_landuse[["other_tags", "shop"]].astype(str)

    singapore_landuse.loc[hdb_condition, "manual_tags"] = "street_shops"
    singapore_landuse.loc[mall_condition, "manual_tags"] = "mall"

    df.update(singapore_landuse[["manual_tags"]])

    return df

    # file_path = str(BASE_DATASET_PATH / "geospatial_data/2021_geospatial/commercial_polygons.xlsx")
    # singapore_landuse.to_excel(file_path)

In [20]:
# add a manual_tags column
singapore_2021_gpd["manual_tags"] = None
singapore_2021_gpd.columns

Index(['osm_id', 'osm_way_id', 'name', 'type', 'aeroway', 'amenity',
       'admin_level', 'barrier', 'boundary', 'building', 'craft', 'geological',
       'historic', 'land_area', 'landuse', 'leisure', 'man_made', 'military',
       'natural', 'office', 'place', 'shop', 'sport', 'tourism', 'other_tags',
       'geometry', 'manual_tags'],
      dtype='object')

In [21]:
# commercial column is not descriptive enough. 
# rows where commercial != null AND other_tags contains HDB will be tagged with street_shops
# rows where commerical != nulul AND shop == "mall" will be tagged with mall
singapore_landuse = singapore_2021_gpd[singapore_2021_gpd["landuse"] == "commercial"].copy()

hdb_condition = singapore_landuse["other_tags"].str.contains("HDB", na = False)
mall_condition = singapore_landuse["shop"] == "mall"

singapore_landuse[["other_tags", "shop"]] = singapore_landuse[["other_tags", "shop"]].astype(str)

singapore_landuse.loc[hdb_condition, "manual_tags"] = "street_shops"
singapore_landuse.loc[mall_condition, "manual_tags"] = "mall"

singapore_2021_gpd.update(singapore_landuse[["manual_tags"]])

file_path = str(BASE_DATASET_PATH / "geospatial_data/2021_geospatial/commercial_polygons.xlsx")
singapore_landuse.to_excel(file_path)

singapore_2021_gpd["manual_tags"].unique()

array([None, 'mall', 'street_shops'], dtype=object)

#### Removing the Former Tanjong Pagar Railway Station for column building = train_station

In [14]:
def remove_by_osm_way_id(df, osm_way_id):
    """
    Removes a row from the Geodataframe by its osm_way_id

    Parameters
    ------
    df : Geodataframe
        Geodataframe you want to modify

    osm_way_id : str
        The osm_way_id of the identified object you want to remove

    returns
    ------
    df : Geodataframe
        Modified Geodataframe with the object removed
    """

    df = df[df["osm_way_id"] != osm_way_id]

    return df

In [ ]:
VALID_FILE_TYPES = {"excel", "csv", "gpkg", "xlsx"}

def save_to_file(df, file_type, filepath):
    """
    Save files to a specified location

    Parameters
    ------
    df : Dataframe
        Dataframe that you want to save

    file_type : str
        The type of file you want to save as (csv, excel, gpkg)
    filepath : str
        Location to save the file
    """

    type = str(file_type).lower()
    
    if type not in VALID_FILE_TYPES:
        raise ValueError(f"results: status must be one of {VALID_FILE_TYPES}.")
    
    if type in ["excel", "xlsx"]:
        df.to_excel(filepath)
    
    elif type in ["csv"]:
        df.to_csv(filepath)
    
    else:
        df.to_file(filepath)


In [ ]:
def read_lines_file(year, BASE_DATASET_PATH):
    """
    Reads geospatial package file

    Parameters
    ------
    year : str
        year of the geospatial data you want to read
    BASE_DATASET_PATH : pathlib.PosixPath
        the base filepath pointing to the "dataset" parent directory
    
    returns
    ------
    singapore_gpd : dataframe
        the geopackage dataframe of singapore's geospatial data
    """

    folder_name = "geospatial_data/" + str(year) + "_geospatial/singapore_" + str(year) + "_lines.gpkg"
    singapore_filepath = BASE_DATASET_PATH / folder_name
    singapore_gpd = gpd.read_file(singapore_filepath)
    print(singapore_gpd.head())
    return singapore_gpd

In [22]:
before = singapore_2021_gpd.shape[0]
singapore_2021_gpd = singapore_2021_gpd[singapore_2021_gpd["osm_way_id"] != "393439098"]
after = singapore_2021_gpd.shape[0]

print(before, after)

125476 125475


In [23]:
# collating the changes and saving to another gpkg file
filepath = str(BASE_DATASET_PATH / "geospatial_data/2021_geospatial/multipolygons_edited_2021.gpkg")
singapore_2021_gpd.to_file(filepath)

In [11]:
singapore_2021_lines_filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/singapore_2021_lines.gpkg"
singapore_2021_lines = gpd.read_file(singapore_2021_lines_filepath)
singapore_2021_lines.crs

<Projected CRS: EPSG:3414>
Name: SVY21 / Singapore TM
Axis Info [cartesian]:
- N[north]: Northing (metre)
- E[east]: Easting (metre)
Area of Use:
- name: Singapore - onshore and offshore.
- bounds: (103.59, 1.13, 104.07, 1.47)
Coordinate Operation:
- name: Singapore Transverse Mercator
- method: Transverse Mercator
Datum: SVY21
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [ ]:

print(f"{singapore_2021_lines.columns} \n")
print(singapore_2021_lines['highway'].unique())

Index(['osm_id', 'name', 'highway', 'waterway', 'aerialway', 'barrier',
       'man_made', 'z_order', 'other_tags', 'geometry'],
      dtype='object') 

['primary' 'residential' 'tertiary' 'footway' 'service' 'secondary'
 'motorway' 'motorway_link' 'trunk' None 'trunk_link' 'primary_link'
 'secondary_link' 'living_street' 'construction' 'pedestrian'
 'unclassified' 'proposed' 'steps' 'track' 'cycleway' 'path'
 'tertiary_link' 'bridleway' 'raceway' 'corridor' 'elevator' 'rest_area']


#### Based on the wiki, I have decided to include motorway, trunk, primary, secondary as the major roads in singapore to include
https://wiki.openstreetmap.org/wiki/WikiProject_Singapore#Roads

In [ ]:
# https://wiki.openstreetmap.org/wiki/WikiProject_Singapore#Roads
major_roads = [
    'motorway', 
    'trunk', 
    'primary', 
    'secondary'
]

def get_major_roads(df, major_roads):
    """
    Get the motorway, trunk, primary and secondary roads of Singapore

    Parameter
    ------
    df : Geodataframe
        Geodataframe containing line layer information

    major_roads : list
        List of road types to be included

    Returns
    ------
    combined_major_roads : Geodataframe
        Geodataframe that containing combined information all the roads of Singapore in one entry
    """

    singapore_major_roads = df[
        df["highway"].isin(major_roads)
    ].copy()

    combined_major_roads = singapore_major_roads.dissolve()

    drop_columns = ["osm_id", "name", "highway", "waterway", "aerialway", "barrier", "man_made", "z_order", "other_tags"]
    combined_major_roads = combined_major_roads.drop(drop_columns, axis = 1, errors = 'ignore')
    combined_major_roads["major_roads"] = "yes"

    return combined_major_roads

In [ ]:
# https://wiki.openstreetmap.org/wiki/WikiProject_Singapore#Roads
major_roads = [
    'motorway', 
    'trunk', 
    'primary', 
    'secondary'
]

singapore_major_roads = singapore_2021_lines[
    singapore_2021_lines['highway'].isin(major_roads)
].copy()

file_path = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/singapore_major_roads_2021.gpkg"
singapore_major_roads.to_file(file_path)

#### Combine the lines of major roads into one singular line
There will be feature loss but the aim was to create a polygon of the major roads. There wasnt a need to preserve the road's features.

In [14]:
combined_major_roads = singapore_major_roads.dissolve()
combined_major_roads.columns
# singapore_2021_gpd.columns

Index(['geometry', 'osm_id', 'name', 'highway', 'waterway', 'aerialway',
       'barrier', 'man_made', 'z_order', 'other_tags'],
      dtype='object')

In [15]:
drop_columns = ["osm_id", "name", "highway", "waterway", "aerialway", "barrier", "man_made", "z_order", "other_tags"]
combined_major_roads = combined_major_roads.drop(drop_columns, axis = 1, errors = 'ignore')
combined_major_roads["major_roads"] = "yes"
print(combined_major_roads)
file_path = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/singapore_combined_major_roads_2021.gpkg"
# # save to gpkg file for processing using qgis
combined_major_roads.to_file(file_path)

                                            geometry major_roads
0  MULTILINESTRING ((27637.517 32038.397, 27643.1...         yes


In [ ]:
# transforming the lines into polygon using the buffer function in qgis
processing.run("native:buffer", {'INPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/singapore_combined_major_roads_2021.gpkg|layername=singapore_combined_major_roads_2021','DISTANCE':5,'SEGMENTS':5,'END_CAP_STYLE':0,'JOIN_STYLE':0,'MITER_LIMIT':2,'DISSOLVE':True,'SEPARATE_DISJOINT':False,'OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/2021_combined_roads_polygon.gpkg'})

### Creating a grid over Singapore using qgis
The grid size will be a hectare large ($100m \times 100m$)

The extents of the grid was determined by looking at Google maps to find the boundaries of Singapore. And converting to 

In [18]:
processing.run("native:creategrid", {'TYPE':2,'EXTENT':'2666.773500000,56466.773500000,15657.950600000,50257.950600000 [EPSG:3414]','HSPACING':100,'VSPACING':100,'HOVERLAY':0,'VOVERLAY':0,'CRS':QgsCoordinateReferenceSystem('EPSG:3414'),'OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/hectare_grid.gpkg'})

{'OUTPUT': '/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/hectare_grid.gpkg'}

```
ogr2ogr -f PostgreSQL \
  PG:"dbname=postgis user=yitong" \
  hectare_grid.gpkg \
  -nln hectare_grid \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  hectare_grid

ogr2ogr -f PostgreSQL \
  PG:"dbname=postgis user=yitong" \
  2021_combined_roads_polygon.gpkg \
  -nln 2021_combined_roads_polygon \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  2021_combined_roads_polygon

ogr2ogr -f PostgreSQL \
  PG:"dbname=postgis user=yitong" \
  multipolygons_edited_2021.gpkg \
  -nln multipolygons_edited_2021 \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  multipolygons_edited_2021
```
#### Fixing invalid geometries
```
UPDATE multipolygons_edited_2021
SET geom = ST_CollectionExtract(ST_MakeValid(geom), 3)::geometry(MultiPolygon, 3414)
WHERE NOT ST_IsValid(geom);
```
#### Intersecting multipolygons with hectare grid
```
DROP TABLE IF EXISTS polygons_grid_2021;

CREATE TABLE polygons_grid_2021 AS
SELECT 
    g.id AS grid_id,
    g.row_index as grid_row,
    g.col_index as grid_col,
    mp.id AS id,
    mp.osm_id as osm_id,
    mp.osm_way_id as osm_way_id,
    mp.name AS name,
	mp.amenity as amenity,
	mp.building as building,
	mp.landuse as landuse,
	mp.military as military,
	mp.office as office,
	mp.place as place,
	mp.shop as shop,
	mp.other_tags as other_tags,
  mp.manual_tags as manual_tags,
    ST_Intersection(ST_MakeValid(mp.geom), g.geom) AS geom
FROM hectare_grid g
JOIN multipolygons_edited_2021 mp
  ON ST_Intersects(g.geom, mp.geom);
```
#### Intersecting road polygons with hectare grid
```
DROP TABLE IF EXISTS roads_grid_2021;

Create TABLE roads_grid_2021 AS
SELECT
  g.id AS grid_id,
  g.row_index as grid_row,
  g.col_index as grid_col,
  mp.major_roads as major_roads,
  ST_Intersection(ST_MakeValid(mp.geom), g.geom) AS geom
  FROM hectare_grid g
  JOIN "2021_combined_roads_polygon" mp
  ON ST_Intersects(g.geom, mp.geom);
```

#### Combining the 2 tables
```
DROP TABLE IF EXISTS combined_grid_2021;
CREATE TABLE combined_grid_2021 AS
SELECT 
    grid_id,
    grid_row,
    grid_col,
    geom,
    major_roads,
    NULL::integer AS id,
    NULL::varchar AS osm_way_id,
    NULL::varchar AS name,
    NULL::varchar AS amenity,
    NULL::varchar AS building,
    NULL::varchar AS landuse,
    NULL::varchar AS military,
    NULL::varchar AS office,
    NULL::varchar AS place,
    NULL::varchar AS shop,
    NULL::varchar AS other_tags,
    NULL::varchar AS manual_tags,
    NULL::varchar AS osm_id
FROM roads_grid_2021

UNION ALL

SELECT 
    grid_id,
    grid_row,
    grid_col,
    geom,
    NULL::text AS major_roads,
    id,
    osm_way_id,
    name,
    amenity,
    building,
    landuse,
    military,
    office,
    place,
    shop,
    other_tags,
    manual_tags,
    osm_id
FROM polygons_grid_2021;
```

The below command shows that the newly created table does not have a primary key
```
SELECT
    kcu.column_name
FROM
    information_schema.table_constraints AS tc
JOIN
    information_schema.key_column_usage AS kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
WHERE
    tc.table_name = 'polygons_grid_2021'
    AND tc.table_schema = 'public'
    AND tc.constraint_type = 'PRIMARY KEY';
```
#### creating a primary key with uid as the column name
```
ALTER TABLE polygons_grid_2021
ADD COLUMN uid SERIAL PRIMARY KEY;
```



In [ ]:
filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/polygon_grids_testing.gpkg"
combined_2021 = gpd.read_file(filepath)
print("Done")

Done


In [88]:
shorten_combined_2021 = combined_2021[
    (combined_2021["grid_id"] <= 50000)
    # & (combined_2021["grid_id"] > 39265)
].sort_values(by="grid_id")
filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/polygon_grids_testing_shorten.gpkg"
shorten_combined_2021.to_file(filepath)
shorten_combined_2021.shape

(205668, 18)

In [ ]:
filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/polygon_grids_testing_shorten.gpkg"
shorten_combined_2021 = gpd.read_file(filepath)
# shorten_combined_2021.head()
print("Done")

In [17]:
# to keep track of the polygon areas of each hectare
hectare_objects = {"airport": 0, "mall": 0, "school": 0, "public_transport_station": 0,
             "major_road": 0, "residential": 0, "workforce": 0, "street_shops": 0}

CHANGI_AIRPORT = "174037464"
SELETAR_AIRPORT = "381760619"
AIRPORT = "airport"
MALL = "mall"
MAJOR_ROAD = "major_road"
STREET_SHOP = "street_shops"
SCHOOL = "school"
RESIDENTIAL = "residential"
WORKFORCE = "workforce"

In [91]:
print(shorten_combined_2021.columns)

Index(['grid_id', 'grid_row', 'grid_col', 'major_roads', 'id', 'osm_way_id',
       'name', 'amenity', 'building', 'landuse', 'military', 'office', 'place',
       'shop', 'other_tags', 'manual_tags', 'osm_id', 'geometry'],
      dtype='object')


[0'grid_id', 1'grid_row', 2'grid_col', 3'major_roads', 4'id', 5'osm_way_id',
       6'name', 7'amenity', 8'building', 9'landuse', 10'military', 11'office', 12'place',
       13'shop', 14'other_tags', 15'manual_tags', 16'osm_id', 17'geometry']

'mall', 'street_shops'

In [92]:
def calculate_area(row, temp_dict, classification):
    if classification == AIRPORT:
        # if the hectare contains airport, that hectare will be classified as airport
        # so make the airport area be the largest
        temp_dict[AIRPORT] += 10000.0
        return temp_dict
    
    if classification == MALL:
        temp_dict[MALL] += 10000.0
        return temp_dict
    
    # if not airport or mall, add up the area of that classification
    temp_dict[classification] += row.geometry.area
    return temp_dict

def decide_classification(current_grid_id, temp_dict, results_pd):
    if all(value == 0 for value in temp_dict.values()):
        classification = None

    else:
        # select the classification with the biggest area
        classification = max(temp_dict, key = temp_dict.get)

    # print(temp_dict)

    grid_id = current_grid_id[0]
    grid_row = current_grid_id[1]
    grid_col = current_grid_id[2]

    # add to the dataframe
    results_pd.loc[len(results_pd)] = [grid_id, grid_row, grid_col, classification]
    

In [93]:
def classify_hectare(input_pd, results_pd):

    current_grid_id = [0, 0, 0]
    temp_dict = hectare_objects.copy()

    for r in input_pd.itertuples(index=False):
        # if the grid_id incremented, means we moving onto the next hectare grid
        if current_grid_id[0] != r.grid_id and current_grid_id[0] != 0:
            decide_classification(current_grid_id, temp_dict, results_pd)

            # reset temp_dict
            temp_dict = hectare_objects.copy()

        # keep track of the current grid id
        current_grid_id = [r.grid_id, r.grid_row, r.grid_col]

        # if the hectare contains airport polygons
        if r.osm_way_id == CHANGI_AIRPORT or r.osm_way_id == SELETAR_AIRPORT:
            temp_dict = calculate_area(r, temp_dict, AIRPORT)
        
        ## As I would want the manual_tags column to override 
        ## the classification from the other columns (mainly for malls and street shops),
        ## they will be checked for first

        # check for malls
        if r.manual_tags == MALL or r.shop == MALL or r.shop == "supermarket" or r.landuse == "retail":
            temp_dict = calculate_area(r, temp_dict, MALL)
            # move to next iteration, manual_tags column should override the classification decision for that row
            continue
        
        # check for street shops
        if r.manual_tags == STREET_SHOP:
            temp_dict = calculate_area(r, temp_dict, STREET_SHOP)
            # move to next iteration, manual_tags column should override the classification decision for that row
            continue

        # check for major roads. major_roads column will be "yes"
        if r.major_roads == "yes":
            temp_dict = calculate_area(r, temp_dict, MAJOR_ROAD)
        
        # check for schools  
        # (primary, secondary, Junior College, polytechnic, universities, childcare)
        if str(r.amenity).lower() in (SCHOOL, "university", "college", "childcare"):
            temp_dict = calculate_area(r, temp_dict, SCHOOL)

        # check for residential area (HDB, condo, landed) 
        if r.landuse == RESIDENTIAL or r.place == "neighbourhood":
            temp_dict = calculate_area(r, temp_dict, RESIDENTIAL)

        # check for workforce
        if str(r.building).lower() in ["industrial", "construction"] or pd.notna(r.office) or r.landuse == "industrial":
            temp_dict = calculate_area(r, temp_dict, WORKFORCE)

    # classify the last hectare manually
    decide_classification(current_grid_id, temp_dict, results_pd)

    return results_pd 


In [98]:
columns_list = ["grid_id", "grid_row", "grid_col", "hectare_classification"]
# for working with classifying of each hectare
results_pd = pd.DataFrame(columns = columns_list)

output_pd = classify_hectare(shorten_combined_2021, results_pd)
output_pd.shape

(36569, 4)

In [100]:
filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/testing_result_classification.xlsx"
output_pd.to_excel(filepath)

In [101]:
columns_list = ["grid_id", "grid_row", "grid_col", "hectare_classification"]
# for working with classifying of each hectare
results_pd = pd.DataFrame(columns = columns_list)

output_pd = classify_hectare(combined_2021, results_pd)
output_pd.shape

(286883, 4)

In [102]:
filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/result_classification.xlsx"
output_pd.to_excel(filepath)

In [81]:
# for r in shorten_combined_2021.itertuples(index=False):
#     print(r.landuse)
for r in shorten_combined_2021.itertuples(index=False):
    # print(r)
    print(r.building, type(r.building))
    # print(r.office, type(r.office))

yes <class 'str'>
yes <class 'str'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
yes <class 'str'>
None <class 'NoneType'>
yes <class 'str'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
yes <class 'str'>
None <class 'NoneType'>
None <class 'NoneType'>
yes <class 'str'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
yes <class 'str'>
yes <class 'str'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'>
yes <class 'str'>
None <class 'NoneType'>
None <class 'NoneType'>
None <class 'NoneType'